# BiLSTM Seq2Seq — Akkadian → English Machine Translation
**Baseline model** with encoder-decoder architecture, teacher forcing, and evaluation via BLEU + chrF++.

In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.model_selection import train_test_split
from sacrebleu import corpus_bleu, corpus_chrf
from tqdm import tqdm
import random
import numpy as np

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

## 1. Load & Clean Data

In [2]:
df = pd.read_csv("data.csv")
df = df[['source', 'translation']].dropna()

def clean_text(text):
    text = text.lower()
    text = text.replace('"', '')
    text = text.replace('<gap>', ' <sep> ')
    return text

df['source'] = df['source'].apply(clean_text)
df['translation'] = df['translation'].apply(clean_text)

# Add special tokens to target
df['translation'] = "<sos> " + df['translation'] + " <eos>"

print(f"Total samples: {len(df)}")
print("\nSample source:", df.iloc[0]['source'])
print("Sample target:", df.iloc[0]['translation'])

Total samples: 62196

Sample source: train
Sample target: <sos> from enna-suen to ennam-aššur: my dear brother, my message came to itūr-ilī there concerning the matter where i have been treated badly. you too should turn to itūr-ilī and assist him, and if they have called a lawsuit  <sep>  do me a great favour. <eos>


## 2. Train / Validation / Test Split

Split **before** building vocabulary to avoid data leakage.  
Split: 80% train, 10% validation, 10% test.

In [3]:
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED)

# Reset indices
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 49756 | Val: 6220 | Test: 6220


## 3. Tokenisation & Vocabulary

Vocabulary is built **only from the training set**.  
`<pad>`, `<unk>`, `<sos>`, `<eos>` are always reserved regardless of frequency.

In [4]:
def tokenize(text):
    return text.split()

for split_df in [train_df, val_df, test_df]:
    split_df['src_tokens'] = split_df['source'].apply(tokenize)
    split_df['tgt_tokens'] = split_df['translation'].apply(tokenize)

def build_vocab(token_lists, min_freq=2):
    """
    Build vocabulary from token lists.
    Special tokens are always included at fixed indices:
      0: <pad>, 1: <unk>, 2: <sos>, 3: <eos>
    """
    counter = Counter()
    for tokens in token_lists:
        counter.update(tokens)

    # Reserve special tokens at fixed positions
    vocab = {"<pad>": 0, "<unk>": 1, "<sos>": 2, "<eos>": 3}

    for word, freq in counter.items():
        if freq >= min_freq and word not in vocab:
            vocab[word] = len(vocab)

    return vocab

# Build vocab from TRAIN only
src_vocab = build_vocab(train_df['src_tokens'])
tgt_vocab = build_vocab(train_df['tgt_tokens'])
tgt_inv_vocab = {v: k for k, v in tgt_vocab.items()}

print(f"Source vocab size: {len(src_vocab)}")
print(f"Target vocab size: {len(tgt_vocab)}")

Source vocab size: 9
Target vocab size: 34633


In [5]:
def encode(tokens, vocab):
    return [vocab.get(t, vocab["<unk>"]) for t in tokens]

for split_df in [train_df, val_df, test_df]:
    split_df['src_ids'] = split_df['src_tokens'].apply(lambda x: encode(x, src_vocab))
    split_df['tgt_ids'] = split_df['tgt_tokens'].apply(lambda x: encode(x, tgt_vocab))

## 4. Dataset & DataLoader

In [6]:
class TranslationDataset(Dataset):
    def __init__(self, df):
        self.src = df['src_ids'].tolist()
        self.tgt = df['tgt_ids'].tolist()

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return torch.tensor(self.src[idx]), torch.tensor(self.tgt[idx])


def collate_fn(batch):
    src, tgt = zip(*batch)
    src = pad_sequence(src, padding_value=0)  # (src_len, batch)
    tgt = pad_sequence(tgt, padding_value=0)  # (tgt_len, batch)
    return src, tgt


train_dataset = TranslationDataset(train_df)
val_dataset   = TranslationDataset(val_df)
test_dataset  = TranslationDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

Train batches: 1555 | Val batches: 195 | Test batches: 195


## 5. Model — BiLSTM Encoder / LSTM Decoder

- **Encoder**: bidirectional LSTM; forward+backward hidden/cell states are concatenated and projected to decoder hidden size.
- **Decoder**: unidirectional LSTM with teacher forcing.
- **Dropout** added to both encoder and decoder embeddings + LSTM for regularisation.

In [7]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=0)
        self.dropout   = nn.Dropout(dropout)
        self.lstm      = nn.LSTM(emb_dim, hid_dim, bidirectional=True, batch_first=False)
        self.fc_hidden = nn.Linear(hid_dim * 2, hid_dim)
        self.fc_cell   = nn.Linear(hid_dim * 2, hid_dim)

    def forward(self, src):
        # src: (src_len, batch)
        embedded = self.dropout(self.embedding(src))  # (src_len, batch, emb_dim)
        outputs, (hidden, cell) = self.lstm(embedded)
        # Concatenate last forward and backward hidden/cell states
        hidden = torch.tanh(self.fc_hidden(torch.cat((hidden[-2], hidden[-1]), dim=1))).unsqueeze(0)
        cell   = torch.tanh(self.fc_cell(  torch.cat((cell[-2],   cell[-1]),   dim=1))).unsqueeze(0)
        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=0)
        self.dropout   = nn.Dropout(dropout)
        self.lstm      = nn.LSTM(emb_dim, hid_dim)
        self.fc        = nn.Linear(hid_dim, output_dim)

    def forward(self, input, hidden, cell):
        # input: (batch,)  →  unsqueeze to (1, batch)
        input    = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))          # (1, batch, emb_dim)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc(output.squeeze(0))                 # (batch, output_dim)
        return prediction, hidden, cell


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = tgt.shape[1]
        tgt_len    = tgt.shape[0]
        vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(tgt_len, batch_size, vocab_size).to(self.device)
        hidden, cell = self.encoder(src)

        input = tgt[0]  # <sos> token
        for t in range(1, tgt_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output
            top1  = output.argmax(1)
            input = tgt[t] if torch.rand(1).item() < teacher_forcing_ratio else top1

        return outputs

## 6. Initialise Model, Optimiser & Loss

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

EMB_DIM = 256
HID_DIM = 512
DROPOUT = 0.5

encoder = Encoder(len(src_vocab), EMB_DIM, HID_DIM, DROPOUT)
decoder = Decoder(len(tgt_vocab), EMB_DIM, HID_DIM, DROPOUT)
model   = Seq2Seq(encoder, decoder, device).to(device)

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=0)  # ignore <pad>

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

Using device: cuda
Trainable parameters: 32,415,561


## 7. Inference Helper

In [9]:
def translate_sentence(model, sentence, src_vocab, tgt_inv_vocab, device, max_len=50):
    """Greedy decode a single source sentence to English."""
    model.eval()
    tokens     = sentence.split()
    src_ids    = [src_vocab.get(t, src_vocab["<unk>"]) for t in tokens]
    src_tensor = torch.tensor(src_ids).unsqueeze(1).to(device)  # (src_len, 1)

    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)

    input_token = torch.tensor([tgt_vocab["<sos>"]]).to(device)
    outputs     = []

    for _ in range(max_len):
        with torch.no_grad():
            output, hidden, cell = model.decoder(input_token, hidden, cell)
        pred_token = output.argmax(1).item()
        word       = tgt_inv_vocab.get(pred_token, "<unk>")
        if word == "<eos>":
            break
        outputs.append(word)
        input_token = torch.tensor([pred_token]).to(device)

    return " ".join(outputs)

## 8. Evaluation Function

Runs on a full DataFrame split and returns BLEU and chrF++ scores.

In [10]:
def evaluate_model(model, df_split, src_vocab, tgt_vocab, tgt_inv_vocab, device):
    """
    Evaluate model on a dataset split.
    Returns BLEU, chrF++ (sacrebleu corpus-level scores).
    """
    preds = []
    refs  = []

    for _, row in df_split.iterrows():
        src = row['source']
        # Strip <sos>/<eos> to get the clean reference
        ref = row['translation'].replace("<sos> ", "").replace(" <eos>", "")

        pred = translate_sentence(model, src, src_vocab, tgt_inv_vocab, device)

        # Normalise <sep> marker consistently in both pred and ref
        pred = pred.replace("<sep>", " ").strip()
        ref  = ref.replace("<sep>", " ").strip()

        preds.append(pred)
        refs.append(ref)

    bleu = corpus_bleu(preds, [refs]).score
    chrf = corpus_chrf(preds, [refs]).score  # chrF++ (word_order=2 by default in sacrebleu)

    return bleu, chrf

## 9. Training Loop

- Gradient clipping (`max_norm=1.0`) to prevent exploding gradients
- Validation loss tracked every epoch
- Best model checkpoint saved based on **validation loss**
- BLEU + chrF++ logged on validation set every epoch

In [11]:
N_EPOCHS   = 30
CLIP       = 1.0
BEST_PATH  = "bilstm_best.pt"

best_val_loss = float('inf')
history = []

for epoch in range(N_EPOCHS):

    # ── Training ──────────────────────────────────────────────
    model.train()
    train_loss = 0

    for src, tgt in tqdm(train_loader, desc=f"Epoch {epoch+1}/{N_EPOCHS} [Train]"):
        src, tgt = src.to(device), tgt.to(device)

        optimizer.zero_grad()
        output     = model(src, tgt)                     # (tgt_len, batch, vocab)
        output_dim = output.shape[-1]

        # Flatten, skipping the <sos> at position 0
        output = output[1:].view(-1, output_dim)
        tgt    = tgt[1:].reshape(-1)

        loss = criterion(output, tgt)
        loss.backward()

        # Gradient clipping — prevents exploding gradients in LSTMs
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)

        optimizer.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # ── Validation loss ───────────────────────────────────────
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for src, tgt in val_loader:
            src, tgt   = src.to(device), tgt.to(device)
            output     = model(src, tgt, teacher_forcing_ratio=0.0)  # no teacher forcing at eval
            output_dim = output.shape[-1]
            output     = output[1:].view(-1, output_dim)
            tgt_flat   = tgt[1:].reshape(-1)
            val_loss  += criterion(output, tgt_flat).item()

    avg_val_loss = val_loss / len(val_loader)

    # ── BLEU / chrF++ on validation set ───────────────────────
    val_bleu, val_chrf = evaluate_model(
        model, val_df, src_vocab, tgt_vocab, tgt_inv_vocab, device
    )

    history.append({
        'epoch': epoch + 1,
        'train_loss': avg_train_loss,
        'val_loss': avg_val_loss,
        'val_bleu': val_bleu,
        'val_chrf': val_chrf
    })

    print(f"\nEpoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} "
          f"| Val BLEU: {val_bleu:.2f} | Val chrF++: {val_chrf:.2f}")

    # ── Checkpoint best model ─────────────────────────────────
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), BEST_PATH)
        print(f"  ✅ New best model saved (val_loss={best_val_loss:.4f})")

print("\nTraining complete.")

Epoch 1/30 [Train]:   0%|          | 7/1555 [06:17<23:12:38, 53.98s/it]


KeyboardInterrupt: 

## 10. Final Evaluation on Test Set

Load the **best checkpoint** and evaluate on the held-out test set.

In [ ]:
# Load best model weights
model.load_state_dict(torch.load(BEST_PATH, map_location=device))

test_bleu, test_chrf = evaluate_model(
    model, test_df, src_vocab, tgt_vocab, tgt_inv_vocab, device
)

print("=" * 40)
print("  TEST SET RESULTS (BiLSTM Baseline)")
print("=" * 40)
print(f"  BLEU   : {test_bleu:.2f}")
print(f"  chrF++ : {test_chrf:.2f}")
print("=" * 40)

## 11. Training History

In [ ]:
history_df = pd.DataFrame(history)
print(history_df.to_string(index=False))